In [7]:
#!/usr/bin/env python3
"""
CORRECTED Text Simplification for Dyslexia-Friendly Content
Fixed version addressing the issues in the previous script
"""

import torch
import pandas as pd
import numpy as np
import re
from transformers import (
    BartForConditionalGeneration, 
    BartTokenizer,
    TrainingArguments,
    Trainer,
    DataCollatorForSeq2Seq
)
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Device setup
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"🍎 Using device: {device}")

# IMPROVED Dyslexia-friendly word replacements (more careful)
WORD_REPLACEMENTS = {
    'hydrologic cycle': 'water cycle',
    'continuous': 'ongoing',
    'instantaneous': 'immediate',
    'systematic': 'careful',
    'examination': 'study',
    'various': 'different',
    'essential': 'important',
    'maintaining': 'keeping',
    'specialized': 'special',
    'cellular organelles': 'cell parts',
    'homeostasis': 'balance',
    'photosynthesis': 'how plants make food',
    'distinct phases': 'separate steps',
    'light-dependent reactions': 'reactions that need light',
    'Calvin cycle': 'Calvin cycle (a process in plants)',
    'characterization': 'describing',
    'symbolism': 'hidden meanings',
    'thematic': 'theme-related',
    'derivative': 'rate of change',
    'represents': 'shows',
    'occurs': 'happens',
    'including': 'such as',
    'requires': 'needs'
}

class FixedSimplificationDataset(Dataset):
    """Fixed dataset without problematic preprocessing"""
    
    def __init__(self, dataframe, tokenizer, max_length=512):
        self.data = dataframe.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length
        
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        # Clean input/output texts - NO INSTRUCTION PREFIX during training
        source_text = str(row['original_text']).strip()
        target_text = str(row['simplified_text']).strip()
        
        # Apply basic word replacements to target only
        for complex_phrase, simple_phrase in WORD_REPLACEMENTS.items():
            target_text = re.sub(r'\b' + re.escape(complex_phrase) + r'\b', 
                               simple_phrase, target_text, flags=re.IGNORECASE)
        
        # Tokenize source (input)
        source = self.tokenizer(
            source_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Tokenize target (output)
        target = self.tokenizer(
            target_text,
            max_length=self.max_length,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Prepare labels
        labels = target['input_ids'].clone()
        labels[labels == self.tokenizer.pad_token_id] = -100
        
        return {
            'input_ids': source['input_ids'].flatten(),
            'attention_mask': source['attention_mask'].flatten(),
            'labels': labels.flatten()
        }

def create_better_training_data():
    """Create high-quality training examples that match your desired output"""
    
    better_examples = [
        {
            'original_text': 'The water cycle, also known as the hydrologic cycle, describes the continuous movement of water on, above, and below the surface of the Earth.',
            'simplified_text': 'The water cycle shows how water moves around Earth. Water moves above, on, and below the ground.',
            'subject': 'science'
        },
        {
            'original_text': 'Water can change states among liquid, vapor, and ice at various places in the water cycle.',
            'simplified_text': 'Water can be liquid, gas (vapor), or solid (ice) in different places.',
            'subject': 'science'
        },
        {
            'original_text': 'The water moves from one reservoir to another, such as from river to ocean, or from the ocean to the atmosphere, by the physical processes of evaporation, condensation, precipitation, infiltration, surface runoff, and subsurface flow.',
            'simplified_text': 'Water moves from place to place in these ways: • Evaporation: Water turns into vapor and rises • Condensation: Vapor turns into clouds • Precipitation: Rain or snow falls • Water flows on the surface and underground',
            'subject': 'science'
        },
        {
            'original_text': 'When water evaporates, it takes up energy from its surroundings and cools the environment. When it condenses, it releases energy and warms the environment.',
            'simplified_text': 'When water evaporates, it makes the area cooler. When it condenses, it makes the area warmer.',
            'subject': 'science'
        },
        {
            'original_text': 'The derivative of a function represents the instantaneous rate of change at any given point.',
            'simplified_text': 'A derivative shows how fast something changes at one exact moment.',
            'subject': 'math'
        },
        {
            'original_text': 'Cellular organelles perform specialized functions essential for maintaining cellular homeostasis.',
            'simplified_text': 'Cell parts do special jobs to keep cells healthy and balanced.',
            'subject': 'science'
        },
        {
            'original_text': 'Literary analysis requires the systematic examination of various literary elements including characterization, symbolism, and thematic development.',
            'simplified_text': 'To study literature, you need to look at: • How characters are described • Hidden meanings in the story • The main themes or messages',
            'subject': 'english'
        },
        {
            'original_text': 'Photosynthesis occurs in two distinct phases: the light-dependent reactions and the light-independent Calvin cycle.',
            'simplified_text': 'Plants make food in two steps: • Light reactions: These need sunlight • Calvin cycle: This can happen without direct light',
            'subject': 'science'
        }
    ]
    
    return pd.DataFrame(better_examples)

def setup_bart_model():
    """Setup BART model (better for text simplification than T5)"""
    print("🤖 Loading BART model (keeping it simple and effective)...")
    
    model_name = "facebook/bart-base"
    tokenizer = BartTokenizer.from_pretrained(model_name)
    model = BartForConditionalGeneration.from_pretrained(model_name)
    
    model = model.to(device)
    
    print(f"Model parameters: {model.num_parameters():,}")
    return model, tokenizer

def improved_post_process(text):
    """Improved post-processing that actually works"""
    
    # Clean up any artifacts
    text = text.strip()
    
    # Apply word replacements
    for complex_phrase, simple_phrase in WORD_REPLACEMENTS.items():
        text = re.sub(r'\b' + re.escape(complex_phrase) + r'\b', 
                     simple_phrase, text, flags=re.IGNORECASE)
    
    # Break very long sentences (over 15 words)
    sentences = re.split(r'[.!?]+', text)
    improved_sentences = []
    
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
            
        words = sentence.split()
        
        # If sentence is too long, try to break it
        if len(words) > 15:
            # Look for natural break points
            if ' and ' in sentence and ', ' in sentence:
                # Convert comma-separated lists to bullet points
                if sentence.count(',') >= 2:
                    parts = sentence.split(':')
                    if len(parts) == 2:
                        intro = parts[0].strip()
                        items = parts[1].split(',')
                        if len(items) >= 2:
                            improved_sentences.append(intro + ':')
                            for item in items:
                                item = item.strip().replace(' and ', '').strip()
                                if item and len(item) > 2:
                                    improved_sentences.append(f"• {item}")
                            continue
            
            # If no natural list, just break at 'and' or 'because'
            if ' and ' in sentence:
                parts = sentence.split(' and ', 1)
                improved_sentences.append(parts[0].strip())
                if len(parts) > 1 and parts[1].strip():
                    improved_sentences.append(parts[1].strip())
            elif ' because ' in sentence:
                parts = sentence.split(' because ', 1)
                improved_sentences.append(parts[0].strip())
                if len(parts) > 1 and parts[1].strip():
                    improved_sentences.append(f"This happens because {parts[1].strip()}")
            else:
                improved_sentences.append(sentence)
        else:
            improved_sentences.append(sentence)
    
    # Join sentences back together
    result = '. '.join([s for s in improved_sentences if s and not s.startswith('•')])
    
    # Add bullet points back
    bullets = [s for s in improved_sentences if s.startswith('•')]
    if bullets:
        result = result + '\n' + '\n'.join(bullets)
    
    # Clean up extra periods
    result = re.sub(r'\.+', '.', result)
    result = re.sub(r'\.\s*$', '', result)  # Remove trailing period
    
    return result

def create_simple_trainer(model, tokenizer, train_dataset, val_dataset):
    """Simpler trainer configuration"""
    
    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer,
        model=model,
        padding=True
    )
    
    # Better training arguments for larger datasets
    training_args = TrainingArguments(
        output_dir='./dyslexia_bart_your_data',
        num_train_epochs=5,  # More epochs since you have real data
        per_device_train_batch_size=4,
        per_device_eval_batch_size=4,
        warmup_steps=100,  # More warmup for larger dataset
        weight_decay=0.01,
        learning_rate=3e-5,  # Good learning rate for fine-tuning
        logging_dir='./dyslexia_bart_logs',
        logging_steps=20,
        eval_strategy="steps",
        eval_steps=50,
        save_strategy="steps",
        save_steps=100,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=3,
        dataloader_pin_memory=False,
        dataloader_num_workers=0,
        fp16=False,
        gradient_checkpointing=False,
        report_to=[],
        prediction_loss_only=False,
        remove_unused_columns=False
    )
    
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset,
        tokenizer=tokenizer,
        data_collator=data_collator
    )
    
    return trainer

def test_fixed_model(model, tokenizer):
    """Test the fixed model"""
    print("\n🧪 Testing fixed dyslexia-friendly model...")
    
    model.eval()
    
    test_examples = [
        "The water cycle, also known as the hydrologic cycle, describes the continuous movement of water on, above, and below the surface of the Earth.",
        "Water can change states among liquid, vapor, and ice at various places in the water cycle.",
        "The derivative of a function represents the instantaneous rate of change at any given point.",
        "Cellular organelles perform specialized functions essential for maintaining cellular homeostasis.",
        "Photosynthesis occurs in two distinct phases: the light-dependent reactions and the Calvin cycle."
    ]
    
    for i, text in enumerate(test_examples):
        print(f"\n--- Test {i+1} ---")
        print(f"Original: {text}")
        
        # NO instruction prefix - just the original text
        inputs = tokenizer(
            text,
            max_length=512,
            truncation=True,
            padding=True,
            return_tensors='pt'
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_length=120,
                min_length=15,
                num_beams=3,  # Fewer beams for faster, more diverse output
                length_penalty=0.6,
                repetition_penalty=1.1,
                early_stopping=True,
                do_sample=False,
                no_repeat_ngram_size=2
            )
        
        # Decode and post-process
        raw_output = tokenizer.decode(outputs[0], skip_special_tokens=True)
        simplified = improved_post_process(raw_output)
        
        print(f"Simplified: {simplified}")
        
        # Calculate metrics
        orig_words = len(text.split())
        simp_words = len(simplified.replace('•', '').split())
        reduction = ((orig_words - simp_words) / orig_words) * 100
        
        print(f"📊 Word reduction: {reduction:.1f}%")
        print(f"📝 Has bullet points: {'✓' if '•' in simplified else '✗'}")

def main():
    """Fixed main training pipeline using YOUR dataset"""
    print("🎓 BART Text Simplification Using YOUR Dataset")
    print("=" * 60)
    
    try:
        # Load YOUR dataset (primary source)
        print("📊 Loading YOUR dataset...")
        df = pd.read_csv('text_edu_dataset.csv', engine='python', on_bad_lines='skip', encoding='utf-8')
        
        # Clean the data
        print(f"Raw dataset: {len(df)} examples")
        df = df.dropna(subset=['original_text', 'simplified_text'])
        df = df[df['original_text'].str.len() > 10]
        df = df[df['simplified_text'].str.len() > 5]
        
        # Remove duplicates
        df = df.drop_duplicates(subset=['original_text'])
        
        print(f"Cleaned dataset: {len(df)} examples")
        print(f"Subjects in your data: {list(df['subject'].unique()) if 'subject' in df.columns else 'No subject column'}")
        
        # Add just a FEW high-quality examples to guide the model (not replace your data!)
        better_df = create_better_training_data()
        
        # Use YOUR data + a small boost from examples
        df = pd.concat([df, better_df], ignore_index=True)
        print(f"Your dataset + examples: {len(df)} total examples ({len(df)-8} from your CSV)")
        
        # Show sample from your data
        print("\n📝 Sample from your dataset:")
        sample_row = df.iloc[0]
        print(f"Original: {sample_row['original_text'][:100]}...")
        print(f"Simplified: {sample_row['simplified_text'][:100]}...")
        
        print(f"Final dataset size: {len(df)} examples")
        
        # Check subject distribution before splitting
        print("\n📊 Subject distribution:")
        if 'subject' in df.columns:
            subject_counts = df['subject'].value_counts()
            print(subject_counts)
            
            # Only use stratified split if all subjects have at least 2 examples
            min_count = subject_counts.min()
            can_stratify = min_count >= 2
            
            if can_stratify and len(df) > 20:
                print("✅ Using stratified split (preserves subject distribution)")
                train_df, temp_df = train_test_split(
                    df, test_size=0.3, random_state=42, 
                    stratify=df['subject']
                )
                # Check if temp_df also has enough examples per subject for second split
                temp_subject_counts = temp_df['subject'].value_counts()
                if temp_subject_counts.min() >= 2:
                    val_df, test_df = train_test_split(
                        temp_df, test_size=0.5, random_state=42,
                        stratify=temp_df['subject']
                    )
                else:
                    # Second split without stratification
                    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
            else:
                print("⚠️  Some subjects have too few examples, using regular split")
                train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
                val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
        else:
            print("📝 No subject column found, using regular split")
            train_df, temp_df = train_test_split(df, test_size=0.3, random_state=42)
            val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)
        
        print(f"Splits: Train={len(train_df)}, Val={len(val_df)}, Test={len(test_df)}")
        
        # Setup BART model (NOT T5)
        model, tokenizer = setup_bart_model()
        
        # Create datasets
        train_dataset = FixedSimplificationDataset(train_df, tokenizer)
        val_dataset = FixedSimplificationDataset(val_df, tokenizer)
        
        # Create trainer
        trainer = create_simple_trainer(model, tokenizer, train_dataset, val_dataset)
        
        # Train model
        print("🚀 Starting training...")
        trainer.train()
        
        # Save model
        print("💾 Saving model...")
        model.save_pretrained('./dyslexia_bart_your_data')
        tokenizer.save_pretrained('./dyslexia_bart_your_data')
        
        # Test model
        test_fixed_model(model, tokenizer)
        
        print("\n✅ Training completed using YOUR dataset!")
        print(f"📁 Model saved to: ./dyslexia_bart_your_data")
        print(f"📊 Trained on {len(train_df)} examples from your CSV")
        
        print("\n💡 To use your trained model:")
        print("from transformers import BartForConditionalGeneration, BartTokenizer")
        print("model = BartForConditionalGeneration.from_pretrained('./dyslexia_bart_your_data')")
        print("tokenizer = BartTokenizer.from_pretrained('./dyslexia_bart_your_data')")
        
    except Exception as e:
        print(f"❌ Error: {e}")
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    print("📦 Required: pip install transformers torch pandas scikit-learn")
    main()

🍎 Using device: mps
📦 Required: pip install transformers torch pandas scikit-learn
🎓 BART Text Simplification Using YOUR Dataset
📊 Loading YOUR dataset...
Raw dataset: 312 examples
Cleaned dataset: 184 examples
Subjects in your data: ['Mathematics', 'Science', 'English']
Your dataset + examples: 192 total examples (184 from your CSV)

📝 Sample from your dataset:
Original: To solve linear equations of the form ax + b = c, where a, b, and c are constants and a ≠ 0, we must...
Simplified: To solve equations like ax + b = c, we need to get x by itself. We do this by doing opposite operati...
Final dataset size: 192 examples

📊 Subject distribution:
subject
Mathematics    70
Science        63
English        51
science         6
math            1
english         1
Name: count, dtype: int64
⚠️  Some subjects have too few examples, using regular split
Splits: Train=134, Val=29, Test=29
🤖 Loading BART model (keeping it simple and effective)...
Model parameters: 139,420,416
🚀 Starting training..

Step,Training Loss,Validation Loss
50,3.505100,2.698602
100,2.624300,2.374942
150,2.344300,2.340306


There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


💾 Saving model...

🧪 Testing fixed dyslexia-friendly model...

--- Test 1 ---
Original: The water cycle, also known as the hydrologic cycle, describes the continuous movement of water on, above, and below the surface of the Earth.
Simplified: The water cycle is the ongoing movement of water on, above,. below the surface of the Earth
📊 Word reduction: 29.2%
📝 Has bullet points: ✗

--- Test 2 ---
Original: Water can change states among liquid, vapor, and ice at various places in the water cycle.
Simplified: Water can change states among liquid, vapor,. ice at different places in the water cycle
📊 Word reduction: 6.2%
📝 Has bullet points: ✗

--- Test 3 ---
Original: The derivative of a function represents the instantaneous rate of change at any given point.
Simplified: The rate of change of a function is the rate of change at any given point
📊 Word reduction: -6.7%
📝 Has bullet points: ✗

--- Test 4 ---
Original: Cellular organelles perform specialized functions essential for maintaining 